In [11]:
from dotenv import load_dotenv

load_dotenv("../.env")
MODEL_ID="jp.anthropic.claude-opus-4-8"

In [12]:
#  asyncio は「イベントループは入れ子（ネスト）にできない」という制約を回避するためのコード
import nest_asyncio
nest_asyncio.apply()

In [13]:
from dataclasses import dataclass

@dataclass
class InventoryItem:
    name: str
    quantity_on_hand: int
    weekly_quantity_sold_past_n_weeks: list[int]
    weeks_to_deliver: int

@dataclass
class Reorder:
    name: str
    quantity_to_order: int
    reason_to_reorder: str

items = [
    InventoryItem("itemA", 300, [50, 70, 80, 100], 2),
    InventoryItem("itemB", 100, [70, 80, 90, 70], 2),
    InventoryItem("itemC", 200, [80, 70, 90, 80], 1)
]

In [ ]:
# https://pydantic.dev/docs/ai/models/bedrock/

from pydantic_ai import Agent

agent = Agent(
    f"bedrock:{MODEL_ID}",
    system_prompt="あなたは、必要な時に必要なだけ発注する在庫管理者です。",
    output_type=list[Reorder]
)

result = agent.run_sync(f"""
これらの品目のうち、今週中に再注文する必要があるものを特定してください。

**Items**
{items}
""")
print(result.output)

[Reorder(name='itemB', quantity_to_order=250, reason_to_reorder='itemBの平均週間販売数は約77.5個（[70,80,90,70]の平均）。配送には2週間かかるため、配送期間中に約155個が販売される見込み。現在の在庫は100個しかなく、配送が完了する前に在庫切れになる。安全在庫を考慮し、配送期間中の需要をカバーしつつ次の補充までの在庫を確保するため再注文が必要。'), Reorder(name='itemC', quantity_to_order=80, reason_to_reorder='itemCの平均週間販売数は約80個（[80,70,90,80]の平均）。配送には1週間かかり、その間に約80個が販売される見込み。現在の在庫は200個で配送期間中はカバーできるが、配送後の在庫が約120個まで減少するため、需要が高まった場合のリスクに備えて今週中に再注文しておくのが望ましい。')]
